<a href="https://colab.research.google.com/github/GideonBature/d2lai/blob/main/chapter_preliminaries/pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing
:label:`sec_pandas`

So far, we have been working with synthetic data
that arrived in ready-made tensors.
However, to apply deep learning in the wild
we must extract messy data
stored in arbitrary formats,
and preprocess it to suit our needs.
Fortunately, the *pandas* [library](https://pandas.pydata.org/)
can do much of the heavy lifting.
This section, while no substitute
for a proper *pandas* [tutorial](https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html),
will give you a crash course
on some of the most common routines.

## Reading the Dataset

Comma-separated values (CSV) files are ubiquitous
for the storing of tabular (spreadsheet-like) data.
In them, each line corresponds to one record
and consists of several (comma-separated) fields, e.g.,
"Albert Einstein,March 14 1879,Ulm,Federal polytechnic school,field of gravitational physics".
To demonstrate how to load CSV files with `pandas`,
we (**create a CSV file below**) `../data/house_tiny.csv`.
This file represents a dataset of homes,
where each row corresponds to a distinct home
and the columns correspond to the number of rooms (`NumRooms`),
the roof type (`RoofType`), and the price (`Price`).


In [5]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('''NumRooms,RoofType,Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000''')

Now let's import `pandas` and load the dataset with `read_csv`.


In [6]:
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


## Data Preparation

In supervised learning, we train models
to predict a designated *target* value,
given some set of *input* values.
Our first step in processing the dataset
is to separate out columns corresponding
to input versus target values.
We can select columns either by name or
via integer-location based indexing (`iloc`).

You might have noticed that `pandas` replaced
all CSV entries with value `NA`
with a special `NaN` (*not a number*) value.
This can also happen whenever an entry is empty,
e.g., "3,,,270000".
These are called *missing values*
and they are the "bed bugs" of data science,
a persistent menace that you will confront
throughout your career.
Depending upon the context,
missing values might be handled
either via *imputation* or *deletion*.
Imputation replaces missing values
with estimates of their values
while deletion simply discards
either those rows or those columns
that contain missing values.

Here are some common imputation heuristics.
[**For categorical input fields,
we can treat `NaN` as a category.**]
Since the `RoofType` column takes values `Slate` and `NaN`,
`pandas` can convert this column
into two columns `RoofType_Slate` and `RoofType_nan`.
A row whose roof type is `Slate` will set values
of `RoofType_Slate` and `RoofType_nan` to 1 and 0, respectively.
The converse holds for a row with a missing `RoofType` value.


In [7]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       NaN           False          True
1       2.0           False          True
2       4.0            True         False
3       NaN           False          True


For missing numerical values,
one common heuristic is to
[**replace the `NaN` entries with
the mean value of the corresponding column**].


In [5]:
inputs = inputs.fillna(inputs.mean())
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       3.0           False          True
1       2.0           False          True
2       4.0            True         False
3       3.0           False          True


## Conversion to the Tensor Format

Now that [**all the entries in `inputs` and `targets` are numerical,
we can load them into a tensor**] (recall :numref:`sec_ndarray`).


In [4]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(targets.to_numpy(dtype=float))
X, y

(tensor([[nan, 0., 1.],
         [2., 0., 1.],
         [4., 1., 0.],
         [nan, 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## Discussion

You now know how to partition data columns,
impute missing variables,
and load `pandas` data into tensors.
In :numref:`sec_kaggle_house`, you will
pick up some more data processing skills.
While this crash course kept things simple,
data processing can get hairy.
For example, rather than arriving in a single CSV file,
our dataset might be spread across multiple files
extracted from a relational database.
For instance, in an e-commerce application,
customer addresses might live in one table
and purchase data in another.
Moreover, practitioners face myriad data types
beyond categorical and numeric, for example,
text strings, images,
audio data, and point clouds.
Oftentimes, advanced tools and efficient algorithms
are required in order to prevent data processing from becoming
the biggest bottleneck in the machine learning pipeline.
These problems will arise when we get to
computer vision and natural language processing.
Finally, we must pay attention to data quality.
Real-world datasets are often plagued
by outliers, faulty measurements from sensors, and recording errors,
which must be addressed before
feeding the data into any model.
Data visualization tools such as [seaborn](https://seaborn.pydata.org/),
[Bokeh](https://docs.bokeh.org/), or [matplotlib](https://matplotlib.org/)
can help you to manually inspect the data
and develop intuitions about
the type of problems you may need to address.


## Exercises

1. Try loading datasets, e.g., Abalone from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets.php) and inspect their properties. What fraction of them has missing values? What fraction of the variables is numerical, categorical, or text?
1. Try indexing and selecting data columns by name rather than by column number. The pandas documentation on [indexing](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html) has further details on how to do this.
1. How large a dataset do you think you could load this way? What might be the limitations? Hint: consider the time to read the data, representation, processing, and memory footprint. Try this out on your laptop. What happens if you try it out on a server?
1. How would you deal with data that has a very large number of categories? What if the category labels are all unique? Should you include the latter?
1. What alternatives to pandas can you think of? How about [loading NumPy tensors from a file](https://numpy.org/doc/stable/reference/generated/numpy.load.html)? Check out [Pillow](https://python-pillow.org/), the Python Imaging Library.


[Discussions](https://discuss.d2l.ai/t/29)


## Solution

Let's work through the exercises in order, explaining each result before checking it with code.

### 1. Inspecting a Dataset and Its Missing Values

Let's use the Abalone file already stored in this repository.
The file has no header row, so we supply the column names ourselves.
`Sex` is a categorical feature, the seven measurements are numerical features,
and `Rings` is a numerical target.
Although `Sex` is stored as strings, its few repeated labels represent categories,
rather than free-form text.

We measure missingness in three ways: the fraction of missing cells,
the fraction of rows containing any missing value,
and the fraction of columns containing any missing value.
These answer different questions; a single missing cell can affect
one row and one column without making the whole dataset mostly missing.

The distributed dataset has 4,177 rows and no missing values.
The [UCI dataset description](https://archive.ics.uci.edu/dataset/1/abalone)
notes that examples with missing values were removed from the original data.
This does not mean all real-world Abalone measurements are complete.

In [8]:
from pathlib import Path
import pandas as pd

# Support running from either the notebook directory or repository root.
data_dir = Path("../data") if Path("../data/abalone.data").exists() else Path("data")
columns = ["Sex", "Length", "Diameter", "Height", "WholeWeight",
           "ShuckedWeight", "VisceraWeight", "ShellWeight", "Rings"]
abalone = pd.read_csv(data_dir / "abalone.data", header=None, names=columns,
                      na_values=["?"])
missing = abalone.isna()
print(abalone.head().to_string(index=False))
print("Shape:", abalone.shape)
print("Missing-cell fraction:", missing.to_numpy().mean())
print("Rows with missing values:", missing.any(axis=1).mean())
print("Columns with missing values:", missing.any(axis=0).mean())
print("Column dtypes:\n", abalone.dtypes)
print("Category labels:", sorted(abalone["Sex"].unique()))
print("Numerical fraction (all columns):", 8 / 9)
print("Categorical fraction (all columns):", 1 / 9)
print("Free-text fraction:", 0.0)

Sex  Length  Diameter  Height  WholeWeight  ShuckedWeight  VisceraWeight  ShellWeight  Rings
  M   0.455     0.365   0.095       0.5140         0.2245         0.1010        0.150     15
  M   0.350     0.265   0.090       0.2255         0.0995         0.0485        0.070      7
  F   0.530     0.420   0.135       0.6770         0.2565         0.1415        0.210      9
  M   0.440     0.365   0.125       0.5160         0.2155         0.1140        0.155     10
  I   0.330     0.255   0.080       0.2050         0.0895         0.0395        0.055      7
Shape: (4177, 9)
Missing-cell fraction: 0.0
Rows with missing values: 0.0
Columns with missing values: 0.0
Column dtypes:
 Sex               object
Length           float64
Diameter         float64
Height           float64
WholeWeight      float64
ShuckedWeight    float64
VisceraWeight    float64
ShellWeight      float64
Rings              int64
dtype: object
Category labels: ['F', 'I', 'M']
Numerical fraction (all columns): 0.88888888888

If we count only the eight input features, the numerical fraction is $7/8$
and the categorical fraction is $1/8$.
Including the target instead gives $8/9$ and $1/9$.
Always state which columns are included when reporting such fractions.

We can also prepare the inputs as in the lesson:
select the features, encode the categorical column, impute numerical values,
and convert the resulting table to a tensor.
For this dataset, imputation leaves the values unchanged because none are missing.

In [9]:
import torch

features = abalone.drop(columns="Rings")
targets = abalone["Rings"]
encoded = pd.get_dummies(features, columns=["Sex"], dummy_na=True, dtype=float)
encoded = encoded.fillna(encoded.mean())
X = torch.tensor(encoded.to_numpy(dtype=float), dtype=torch.float32)
y = torch.tensor(targets.to_numpy(dtype=float), dtype=torch.float32)
print("Feature tensor shape:", X.shape)
print("Target tensor shape:", y.shape)
print("Remaining missing cells:", int(encoded.isna().sum().sum()))

Feature tensor shape: torch.Size([4177, 11])
Target tensor shape: torch.Size([4177])
Remaining missing cells: 0


### 2. Selecting Columns by Name

`iloc` selects by integer position, whereas `loc` selects by labels.
Selecting a single name produces a Series;
selecting a list of names produces a DataFrame.
Named selections remain readable even if column order changes.
The [pandas indexing guide](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html)
explains these two forms of indexing.

Let's reproduce the lesson's split using `house_tiny.csv`.
We can also combine a Boolean row condition with named columns.

In [10]:
houses = pd.read_csv(data_dir / "house_tiny.csv")
inputs_by_name = houses.loc[:, ["NumRooms", "RoofType"]]
targets_by_name = houses.loc[:, "Price"]
print("Features:\n", inputs_by_name)
print("Targets:\n", targets_by_name)
print("Same features as positional selection:", inputs_by_name.equals(houses.iloc[:, :2]))
print("Higher-priced houses:\n",
      houses.loc[houses["Price"] > 130000, ["NumRooms", "Price"]])

Features:
    NumRooms RoofType
0       NaN      NaN
1       2.0      NaN
2       4.0    Slate
3       NaN      NaN
Targets:
 0    127500
1    106000
2    178100
3    140000
Name: Price, dtype: int64
Same features as positional selection: True
Higher-priced houses:
    NumRooms   Price
2       4.0  178100
3       NaN  140000


### 3. How Much Data Can We Load?

There is no single row limit: capacity depends on column types,
available memory, and the operations we perform.
A numeric table of $N$ rows and $d$ float64 columns needs approximately
$8Nd$ bytes for its values alone.
Strings, the index, parser buffers, encoded categories,
and temporary copies can increase the total considerably.
Reading time also depends on storage speed and CSV parsing.

Let's run a bounded experiment on this machine rather than assume its capacity.
We create a temporary CSV containing 100,000 rows and four numeric columns,
measure its read time and DataFrame memory, and compare the estimate
for larger numeric tables. Temporary files are removed automatically.

In [11]:
import numpy as np
from tempfile import TemporaryDirectory
from time import perf_counter

rng = np.random.default_rng(7)
with TemporaryDirectory() as temporary_dir:
    csv_path = Path(temporary_dir) / "numeric_sample.csv"
    sample = pd.DataFrame(rng.normal(size=(100_000, 4)), columns=list("ABCD"))
    sample.to_csv(csv_path, index=False)
    del sample
    start = perf_counter()
    loaded = pd.read_csv(csv_path)
    elapsed = perf_counter() - start
    memory_bytes = loaded.memory_usage(index=True, deep=True).sum()
    print("Rows loaded:", len(loaded))
    print("CSV size (MiB):", round(csv_path.stat().st_size / 2**20, 2))
    print("DataFrame memory (MiB):", round(memory_bytes / 2**20, 2))
    print("Read time (seconds):", round(elapsed, 3))

    # Accumulate a statistic without retaining every chunk at once.
    total, count = 0.0, 0
    for chunk in pd.read_csv(csv_path, usecols=["A"], chunksize=20_000):
        total += chunk["A"].sum()
        count += len(chunk)
    print("Chunked mean agrees:", np.isclose(total / count, loaded["A"].mean()))

for rows in [1_000_000, 10_000_000]:
    print(rows, "rows, 20 float64 columns: value storage (GiB) =",
          round(rows * 20 * 8 / 2**30, 3))

Rows loaded: 100000
CSV size (MiB): 7.49
DataFrame memory (MiB): 3.05
Read time (seconds): 0.037
Chunked mean agrees: True
1000000 rows, 20 float64 columns: value storage (GiB) = 0.149
10000000 rows, 20 float64 columns: value storage (GiB) = 1.49


The measured time describes this machine and this sample, not a universal limit.
The larger-table estimates omit parser and processing overhead.
Exceeding available memory may cause swapping, severe slowdown,
an allocation error, or process termination.
Reading selected columns, choosing narrower dtypes, and processing chunks
can reduce memory requirements.

A server with more RAM, faster storage, or more cores may handle larger tables,
but the same constraints still apply. We have not run this experiment on a server;
a server measurement would require its actual hardware and workload.
Chunking helps only if later steps also avoid collecting all chunks into memory.

### 4. Handling Many Categories and Unique Labels

One-hot encoding creates a column for each distinct category.
With $N$ rows and $K$ categories, dense encoding uses $NK$ entries,
so high-cardinality columns can become expensive.
Sparse one-hot representations store mostly the nonzero entries.
Other choices include grouping rare categories, hashing into a fixed number
of buckets, or learning embeddings when training a model.
Hashing can create collisions; embeddings still require learned parameters.

An arbitrary identifier that is unique for every row usually does not provide
a reusable predictive pattern. Its one-hot columns appear only once each,
so we generally exclude it as a feature.
A meaningful identifier that repeats across observations, such as a product ID,
may still be useful if the evaluation setup matches that use.
Free-form text should be represented according to its content,
rather than treating every distinct sentence as an unrelated category.

Let's see the difference between a repeated category and a unique row ID.

In [12]:
toy = pd.DataFrame({"Color": ["red", "blue", "red", "green"],
                    "RowID": ["id_1", "id_2", "id_3", "id_4"]})
color_encoding = pd.get_dummies(toy[["Color"]], dtype=float)
id_encoding = pd.get_dummies(toy[["RowID"]], dtype=float)
print("Repeated-category encoding shape:", color_encoding.shape)
print("Unique-ID encoding shape:", id_encoding.shape)
print("Unique-ID encoding:\n", id_encoding)
print("Features after dropping the ID:\n", toy.drop(columns="RowID"))

Repeated-category encoding shape: (4, 3)
Unique-ID encoding shape: (4, 4)
Unique-ID encoding:
    RowID_id_1  RowID_id_2  RowID_id_3  RowID_id_4
0         1.0         0.0         0.0         0.0
1         0.0         1.0         0.0         0.0
2         0.0         0.0         1.0         0.0
3         0.0         0.0         0.0         1.0
Features after dropping the ID:
    Color
0    red
1   blue
2    red
3  green


### 5. Alternatives for Other Data Types

The appropriate loader depends on the data format and the operations we need.
NumPy is useful for homogeneous numerical arrays;
its `.npy` format preserves shape and dtype without CSV parsing.
`np.load(..., mmap_mode="r")` can access a stored array through memory mapping,
which helps avoid eagerly reading the entire array into RAM.
Later computations can still allocate substantial memory.
See the [NumPy loading documentation](https://numpy.org/doc/stable/reference/generated/numpy.load.html).

Pillow loads images and can convert them into RGB arrays.
An RGB image becomes an array with shape `(height, width, 3)`;
we can then create a PyTorch tensor and move channels to the first axis.
The [Pillow handbook](https://pillow.readthedocs.io/en/stable/handbook/tutorial.html)
introduces its image-reading and conversion operations.

For tabular data, SQL databases can filter and aggregate records before loading,
and tools such as Polars, Arrow, or Dask offer other processing approaches.
These choices serve different workloads rather than replacing every pandas operation.
Let's demonstrate NumPy and Pillow using small temporary files.

In [13]:
from PIL import Image

with TemporaryDirectory() as temporary_dir:
    array_path = Path(temporary_dir) / "measurements.npy"
    values = np.arange(12, dtype=np.float32).reshape(3, 4)
    np.save(array_path, values)
    mapped = np.load(array_path, mmap_mode="r", allow_pickle=False)
    print("Loaded array shape and dtype:", mapped.shape, mapped.dtype)
    print("Memory-mapped:", isinstance(mapped, np.memmap))
    print("Values agree:", np.array_equal(values, mapped))
    del mapped

    image_path = Path(temporary_dir) / "sample.png"
    pixels = np.zeros((16, 24, 3), dtype=np.uint8)
    pixels[:, :12, 0] = 255
    pixels[:, 12:, 2] = 255
    Image.fromarray(pixels).save(image_path)
    with Image.open(image_path) as image:
        rgb = np.array(image.convert("RGB"))
    image_tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255
    print("Image array shape:", rgb.shape)
    print("Channels-first tensor shape:", image_tensor.shape)

Loaded array shape and dtype: (3, 4) float32
Memory-mapped: True
Values agree: True
Image array shape: (16, 24, 3)
Channels-first tensor shape: torch.Size([3, 16, 24])
